# ProtoPNet

In [ ]:
import os
import hydra
import torch
import shutil
import warnings
from torch.optim import Adam
from omegaconf import OmegaConf
from utils import check_dir
from tes_gnnNets import *
from dataset import get_dataset, get_dataloader
import torch.nn.functional as F
from torch.optim.lr_scheduler import MultiStepLR
import logging
import networkx as nx
import torch_geometric
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

In [ ]:
from hydra import compose, initialize
from omegaconf import OmegaConf
hydra.core.global_hydra.GlobalHydra.instance().clear()
initialize(config_path="tes_config", job_name="test_app")
cfg = compose(config_name="config", overrides=["datasets=ba_2motifs", "seed=0","models.param.ba_2motifs.num_basis_per_class=10"]) #here put the same settings used for the train
#print(OmegaConf.to_yaml(cfg))


In [ ]:
config = cfg

In [ ]:
config.models.gnn_saving_dir = 'tes_checkpoints'
config.models.param = config.models.param[config.datasets.dataset_name]

if torch.cuda.is_available():
    device = torch.device('cuda', index=config.device_id)
else:
    device = torch.device('cpu')

dataset = get_dataset(dataset_root=config.datasets.dataset_root,
                      dataset_name=config.datasets.dataset_name)
dataset.data.x = dataset.data.x.float()
dataset.data.y = dataset.data.y.squeeze().long()
if config.models.param.graph_classification:
    dataloader_params = {'batch_size': 1,
                         'random_split_flag': config.datasets.random_split_flag,
                         'data_split_ratio': config.datasets.data_split_ratio,
                         'seed': config.seed}

In [ ]:
!ls tes_checkpoints

In [ ]:
dataset = get_dataset(dataset_root=config.datasets.dataset_root,
                          dataset_name=config.datasets.dataset_name)

In [ ]:
dataloader = get_dataloader(dataset, **dataloader_params)

In [ ]:
model = get_gnnNets(dataset.num_node_features, dataset.num_classes, config.models)

In [ ]:
f"outputs_tes/datasets={config.datasets.dataset_name},device_id=0,models.param.{config.datasets.dataset_name}.num_basis_per_class=10,models=gcn,seed={config.seed}/tes_checkpoints/{config.seed}/{config.datasets.dataset_name}_from_None/gcn_{config.models.param.num_basis_per_class}_3l_best.pth"

In [ ]:
saved = torch.load(f"outputs_tes/datasets={config.datasets.dataset_name},device_id=0,models.param.{config.datasets.dataset_name}.num_basis_per_class=10,models=gin,seed={config.seed}/tes_checkpoints/{config.seed}/{config.datasets.dataset_name}_from_None/gin_{config.models.param.num_basis_per_class}_5l_best.pth") #here the path to the saved model
state_dict = saved['net']
basis_data = saved['basis_data']
model.load_state_dict(state_dict)
model.basis_data = basis_data

In [ ]:
# !ls "outputs_tes/datasets=ba_2motifs,device_id=0,models.param.ba_2motifs.num_basis_per_class=10,models=gin,seed=0/tes_checkpoints/0/ba_2motifs_from_None"

In [ ]:
def set_config_ex(dataset="ba_2motifs", seed=0, num_basis_per_class=10, config_name="tes", model_name="gin"):
    hydra.core.global_hydra.GlobalHydra.instance().clear()
    initialize(config_path=config_name + "_" + "config" , job_name="test_app")
    cfg = compose(config_name="config", overrides=[f"datasets={dataset}", f"seed={seed}", f"models={model_name}"])
#     print(OmegaConf.to_yaml(cfg))
    config = cfg
    config.models.gnn_saving_dir = config_name + "_" + 'checkpoints'
    config.models.param = config.models.param[config.datasets.dataset_name]

    if torch.cuda.is_available():
        device = torch.device('cuda', index=config.device_id)
    else:
        device = torch.device('cpu')

    dataset = get_dataset(dataset_root=config.datasets.dataset_root,
                          dataset_name=config.datasets.dataset_name)
    dataset.data.x = dataset.data.x.float()
    dataset.data.y = dataset.data.y.squeeze().long()
    if config.models.param.graph_classification:
        dataloader_params = {'batch_size': 1,
                             'random_split_flag': config.datasets.random_split_flag,
                             'data_split_ratio': config.datasets.data_split_ratio,
                             'seed': config.seed}
    dataset = get_dataset(dataset_root=config.datasets.dataset_root,
                          dataset_name=config.datasets.dataset_name)
    dataloader = get_dataloader(dataset, **dataloader_params)
    model = get_gnnNets(dataset.num_node_features, dataset.num_classes, config.models)
    saved = torch.load(f"outputs_tes/datasets={config.datasets.dataset_name},device_id=0,models.param.{config.datasets.dataset_name}.num_basis_per_class=10,models=gin,seed={config.seed}/tes_checkpoints/{config.seed}/{config.datasets.dataset_name}_from_None/gin_{config.models.param.num_basis_per_class}_5l_latest.pth", map_location='cuda:0') #here the path to the saved model
    # saved = torch.load(f"outputs_tes_old/datasets={config.datasets.dataset_name},device_id=0,models.param.{config.datasets.dataset_name}.num_basis_per_class=10,models=gcn,seed={config.seed}/tes_checkpoints/{config.seed}/{config.datasets.dataset_name}_from_None/gcn_{config.models.param.num_basis_per_class}_3l_latest.pth", map_location='cuda:0') #here the path to the saved model
    state_dict = saved['net']
    basis_data = saved['basis_data']
    model.load_state_dict(state_dict)
    model.basis_data = basis_data
    return config, dataset, dataloader, model

In [ ]:
dataset_name='bbbp'

In [ ]:
def explain(model, data, sparsity=0.8, c=None, protop=True):
    out, embs, cosines=model(data.x.float(), data.edge_index)    
    y_hat=c
    if y_hat is None:
        y_hat=out.argmax(-1).item()
    mask=np.zeros(len(data.x))
    if True:
        max_nodes = cosines[:,model.num_basis_per_class*y_hat:model.num_basis_per_class*(y_hat+1)].argmax(axis=0).tolist()
        for i in range(model.num_basis_per_class*y_hat,model.num_basis_per_class*(y_hat+1)):
            try:
                to_mask = torch_geometric.utils.k_hop_subgraph(max_nodes[i%model.num_basis_per_class], 3, data.edge_index)[0].tolist()
                cosines_i = cosines[max_nodes[i%model.num_basis_per_class],i].item()
                mask[to_mask] += model.classifier_weights[i, y_hat].detach().numpy()*cosines_i
            except:pass
    else:
        for i in range(len(data.x)):
            mask[i]+= cosines[i,model.num_basis_per_class*y_hat:model.num_basis_per_class*(y_hat+1)].sum().item()
    # if protop: mask=torch.log(mask+1)/(mask+1e-8)
    # mask = mask*model.classifier_weights[i]
    mask=(mask-mask.min())/(mask.max()-mask.min()+1e-8)
    # p=np.percentile(mask, 100*sparsity)
    # mask[mask<p]=0
    # edge_mask=np.zeros(data.edge_index.shape[1])
    # for i, (a,b) in enumerate(data.edge_index.T):
    #     edge_mask[i]=(mask[a]+mask[b])/2
    # edge_mask[edge_mask>0]=1
    return mask#, edge_mask
    

In [ ]:
import torch
import torch.nn.functional as F
from torch_geometric.utils import subgraph, add_self_loops
from torch_geometric.data import Data,Batch

def calculate_fidelity(data, node_mask, model, remove_nodes=True, top_k=None):
    data = Batch.from_data_list([data])
    device = next(model.parameters()).device
    data = data.to(device)

    # Compute sparsity
    total_nodes = data.x.shape[0]
    sparsity = 1 - node_mask.sum().item() / total_nodes if total_nodes > 0 else 0

    # Get original predictions
    original_pred = model(data)[0]
    original_pred = F.softmax(original_pred, dim=1)
    label = original_pred.argmax(-1).item()
    
    # Apply node mask (InvFidelity)
    if remove_nodes:
        masked_edge_index, _ = subgraph(node_mask == 0, edge_index=data.edge_index, num_nodes=data.x.size(0), relabel_nodes=True)
        n_nodes = (node_mask == 0).sum()
        new_data = Batch.from_data_list([Data(x=data.x[node_mask == 0], edge_index=masked_edge_index)])
        masked_pred = model(new_data)[0]
    else:
        masked_x = data.x.clone()
        masked_x[node_mask==1] = 0
        new_data = Batch.from_data_list([Data(x=masked_x, edge_index=data.edge_index)])
        masked_pred = model(new_data)[0]
    masked_pred = F.softmax(masked_pred, dim=1)

    # Keep only important nodes (Fidelity)
    if remove_nodes:
        masked_edge_index, _ = subgraph(node_mask == 1, edge_index=data.edge_index, num_nodes=data.x.size(0), relabel_nodes=True)
        n_nodes = (node_mask == 1).sum()
        new_data = Batch.from_data_list([Data(x=data.x[node_mask == 1], edge_index=masked_edge_index)])
        retained_pred = model(new_data)[0]
    else:
        masked_x = data.x.clone()
        masked_x[node_mask==0] = 0
        new_data = Batch.from_data_list([Data(x=masked_x, edge_index=data.edge_index)])
        retained_pred = model(new_data)[0]
    retained_pred = F.softmax(retained_pred, dim=1)


    # # Compute Fidelity+ and Fidelity-
    # inv_fidelity = (original_pred[:, label] - 
    #                              retained_pred[:, label]).mean().item()

    # fidelity = (original_pred[:, label] - 
    #                              masked_pred[:, label]).mean().item()

    inv_fidelity = (original_pred.argmax(-1) != 
                                 retained_pred.argmax(-1)).float().item()

    fidelity = (original_pred.argmax(-1)  != 
                                 masked_pred.argmax(-1) ).float().item()

    n_fidelity = inv_fidelity*sparsity
    n_inv_fidelity = inv_fidelity*(1-sparsity)
    
    # Compute HFidelity (harmonic mean of Fidelity+ and Fidelity-)
    hfidelity = ((1+n_fidelity) * (1-n_inv_fidelity)) / (2 + n_fidelity - n_inv_fidelity) if (1 + n_fidelity - n_inv_fidelity) != 0 else 0

    return {
        "Fidelity": fidelity,
        "InvFidelity": inv_fidelity,
        "HFidelity": hfidelity
    }

In [ ]:
# def generate_hard_masks(soft_mask):
#     sparsity_levels = torch.arange(0.5,1, 0.05)
#     hard_masks = []
#     for sparsity in sparsity_levels:
#         threshold = np.percentile(soft_mask, sparsity * 100)
#         hard_mask = (soft_mask > threshold).int()
#         if hard_mask.sum() == 0:
#             hard_mask = (soft_mask > soft_mask.min()).int()
#         hard_masks.append(hard_mask)
#     return list(zip(sparsity_levels, hard_masks))
import torch
import numpy as np

import torch

def generate_hard_masks(soft_mask):
    soft_mask_flat = soft_mask.flatten()
    total_elements = soft_mask_flat.numel()
    sparsity_levels = torch.arange(0.5, 1.0, 0.05)
    hard_masks = []

    # Get sorted indices (ascending: lowest values first)
    sorted_indices = torch.argsort(soft_mask_flat)

    for sparsity in sparsity_levels:
        num_to_mask = int(sparsity.item() * total_elements)
        mask_flat = torch.ones_like(soft_mask_flat, dtype=torch.int)
        
        if num_to_mask >= total_elements:
            num_to_mask = total_elements - 1  # keep at least one element
        
        # Zero out the lowest `num_to_mask` elements
        mask_flat[sorted_indices[:num_to_mask]] = 0
        
        # Reshape to original shape
        hard_mask = mask_flat.view_as(soft_mask)
        hard_masks.append(hard_mask)

    return list(zip(sparsity_levels, hard_masks))



In [ ]:
import tqdm
import pandas as pd

In [ ]:
config, dataset, dataloader, model = set_config_ex(dataset=dataset_name, seed=0, model_name="gin", config_name="tes")
l = []
seeds = [0,1,2,3,4]
dataset_name = 'bbbp'
for i in seeds:
    try:
        config, dataset, dataloader, model = set_config_ex(dataset=dataset_name, seed=i, model_name="gin", config_name="tes")
    except:
        continue
    # preds, accs = [], []
    # for batch in dataloader['test']:
    #     batch = batch
    #     batch_preds, _, _ = model(batch.x.float(), batch.edge_index)
    #     batch_preds = batch_preds.argmax(-1)
    #     preds.append(batch_preds)
    #     accs.append(batch_preds == batch.y)
    # preds = torch.cat(preds, dim=-1)
    # test_acc = torch.cat(accs, dim=-1).float().mean().item()
    seed_expl_res = []
    for data in tqdm.tqdm(dataloader['test']):
        if data.y==0: continue
        try:
            data.x = data.x.float()
            out = model(data=data)
        except Exception as e:
            print(e)
            continue
        logit, embs, sims = out
        c = logit.argmax(-1).item()
        mask = explain(model, data,c=c)
        # except:
        #     continue
        masks = generate_hard_masks(torch.tensor(mask).cpu().detach())
        d = {}
        for m in masks:
            d[m[0].item()] = calculate_fidelity(data, m[1], model)
        seed_expl_res.append(pd.DataFrame(d))
    l.append(sum(seed_expl_res)/len(seed_expl_res))

In [ ]:
from dataset import get_dataset

In [ ]:
l = [np.array(x) for x in l]

In [ ]:
l

In [ ]:
np.mean(l, axis=0)[0]

In [ ]:
np.mean(l)

In [ ]:
np.std(l)

In [ ]:
dataset

In [ ]:
import tqdm
l = []
for data in tqdm.tqdm(dataloader['test']):
    try:
        out = model(data=b)
        logit, embs, sims = out
        c = logit.argmax(-1).item()
        mask = explain(model, data,c=c)
    except:
        continue
    masks = generate_hard_masks(torch.tensor(mask).cpu().detach())
    d = {}
    for m in masks:
        d[m[0].item()] = calculate_fidelity(data, m[1], model)
    l.append(d)

In [ ]:
import pandas as pd
l = [pd.DataFrame(x) for x in l]

In [ ]:
sum(l)/len(l)